# k-Nearest Neighbour para decidir cuál modelo es mejor para cada consulta

In [65]:
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
import joblib
import gower
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score

In [66]:
data = "resultados.json"
data = pd.read_json(data)

In [67]:
print(data.head())

  departamento usuario                                              query  \
0    marketing     ana  Redacta una secuencia de 5 correos electrónico...   
1    marketing     ana  Redacta una secuencia de 5 correos electrónico...   
2    marketing     ana  Redacta una secuencia de 5 correos electrónico...   
3    marketing     ana  Escribe el guion completo para un vídeo de You...   
4    marketing     ana  Escribe el guion completo para un vídeo de You...   

                  model  status_code  latency_s  prompt_tokens  \
0           llama3.2:3b          200   4.952600             70   
1            mistral:7b          200   7.475177             63   
2  llama-3.1-8b-instant          200   0.765976             80   
3           llama3.2:3b          200   1.366815             69   
4            mistral:7b          200   2.777333             60   

   completion_tokens  total_tokens      cost  \
0                 15            85  0.000005   
1                 15            78  0.000019

In [68]:
# Crear el DataFrame con las variables predictoras
datos = data.drop(columns=['model','usuario', 'response_preview', 'query', 'status_code'])
datos['target'] = data['model']

display(datos.head())  

print(f"Número de ejemplos (filas): {datos.shape[0]}")
print(f"Número de variables totales (columnas): {datos.shape[1]}")

columnas_numericas = datos.select_dtypes(include=[np.number]).columns
medianas = datos[columnas_numericas].median()
datos[columnas_numericas] = datos[columnas_numericas].fillna(medianas)
columnas_string_conflictivas = datos.select_dtypes(include=['string']).columns

for col in columnas_string_conflictivas:
    datos[col] = datos[col].astype(object)

,departamento,latency_s,prompt_tokens,completion_tokens,total_tokens,cost,target
0,marketing,4.952600,70,15,85,0.000005,llama3.2:3b
1,marketing,7.475177,63,15,78,0.000019,mistral:7b
2,marketing,0.765976,80,15,95,0.000005,llama-3.1-8b-instant
3,marketing,1.366815,69,15,84,0.000005,llama3.2:3b
4,marketing,2.777333,60,15,75,0.000018,mistral:7b


Número de ejemplos (filas): 180
Número de variables totales (columnas): 7


In [69]:
# Separamos características (X) y objetivo (y)
y = datos['target']
X = datos.drop(columns=['target'])

# Dividimos en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y
)

# Comprobamos la distribución (en porcentaje)
print("Distribución en Entrenamiento:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribución en Prueba:")
print(y_test.value_counts(normalize=True) * 100)

Distribución en Entrenamiento:
target
llama3.2:3b             33.333333
mistral:7b              33.333333
llama-3.1-8b-instant    33.333333
Name: proportion, dtype: float64

Distribución en Prueba:
target
llama3.2:3b             33.333333
llama-3.1-8b-instant    33.333333
mistral:7b              33.333333
Name: proportion, dtype: float64


In [70]:
X_train_dist = gower.gower_matrix(X_train)

# Inicializamos el modelo k-NN con k=3 y distancia de Gower
knn = KNeighborsClassifier(n_neighbors=3, metric='precomputed')

# 1. Identificamos cuáles son las columnas de texto/categóricas
columnas_categoricas = datos.select_dtypes(include=['object', 'string']).columns

# (Asegúrate de calcular tu 'cat_mask' y tus 'rangos' basándote en el 
# DataFrame original ANTES de aplicar la transformación del siguiente paso).

# 2. Creamos una copia de los datos para no machacar los originales
datos_numericos = datos.copy()

# Aplicamos ordinal encoding (ej: 'marketing' -> 0, 'desarrollo' -> 1)
codificador = OrdinalEncoder()
datos_numericos[columnas_categoricas] = codificador.fit_transform(datos[columnas_categoricas])

matriz_distancias_train = gower.gower_matrix(X_train) 
knn = KNeighborsClassifier(n_neighbors=3, metric='precomputed')
knn.fit(matriz_distancias_train, y_train)

print("¡Modelo entrenado superando el bloqueo de texto!")

¡Modelo entrenado superando el bloqueo de texto!


In [71]:
X_test_dist = gower.gower_matrix(X_test, X_train)

# Generamos las predicciones sobre el conjunto de prueba
y_pred = knn.predict(X_test_dist)

print("Matriz de entrenamiento mapeada con éxito.")
print(f"Predicciones del conjunto de prueba: {y_pred}")

# Contamos cuántas predicciones coinciden exactamente con la realidad
aciertos = (y_test == y_pred).sum()
total_casos = len(y_test)

print(f"El modelo ha acertado {aciertos} de un total de {total_casos} casos.")
porcentaje_aciertos = accuracy_score(y_test, y_pred)
print(f"Porcentaje de aciertos: {porcentaje_aciertos * 100:.2f}%")

Matriz de entrenamiento mapeada con éxito.
Predicciones del conjunto de prueba: ['llama3.2:3b' 'llama-3.1-8b-instant' 'llama-3.1-8b-instant' 'llama3.2:3b'
 'llama-3.1-8b-instant' 'mistral:7b' 'llama3.2:3b' 'mistral:7b'
 'mistral:7b' 'mistral:7b' 'llama-3.1-8b-instant' 'llama-3.1-8b-instant'
 'llama3.2:3b' 'mistral:7b' 'mistral:7b' 'llama3.2:3b'
 'llama-3.1-8b-instant' 'llama3.2:3b' 'llama3.2:3b' 'llama3.2:3b'
 'mistral:7b' 'llama3.2:3b' 'llama3.2:3b' 'llama-3.1-8b-instant'
 'llama-3.1-8b-instant' 'llama-3.1-8b-instant' 'llama3.2:3b' 'mistral:7b'
 'llama3.2:3b' 'mistral:7b' 'llama3.2:3b' 'mistral:7b' 'llama3.2:3b'
 'mistral:7b' 'mistral:7b' 'llama3.2:3b']
El modelo ha acertado 25 de un total de 36 casos.
Porcentaje de aciertos: 69.44%


In [72]:
joblib.dump(knn, 'modelo_knn_ej8.joblib')
print("¡Modelo exportado correctamente!")

¡Modelo exportado correctamente!
